# E10 ImageNet-C

Migración de DeMemte desde Flowers-102 hacia ImageNet/ImageNet-C.
`source_resnet50_imagenet` es el baseline de clasificación real; las
variantes DeMemte/E10 usan `num_classes=1000` y el backbone ImageNet.


In [8]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd()
while ROOT.name != "Dememte" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from dememte.config import E6Config
from dememte.data import build_imagenet_c_loader
from dememte.evaluation import evaluate_baseline, evaluate_dememte, evaluate_dememte_tta
from dememte.io import load_checkpoint
from dememte.memory import HippocampalConfig, HippocampalMemoryAdapter
from dememte.models import make_imagenet_resnet50
from dememte.models.dememte import make_dememte_variant


In [9]:
DATA_ROOT = ROOT / "experiments/data/imagenet-c-subset"
OUT_DIR = ROOT / "notebooks/10b_imagenet_c/out"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DEMEMTE_CHECKPOINT = ROOT / "experiments/imagenet_dememte/out/dememte_imagenet_resnet50_vqsa_best.pt"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64
NUM_WORKERS = 4
CORRUPTIONS = ["gaussian_noise", "motion_blur", "pixelate", "jpeg_compression"]
SEVERITIES = [3, 5]
MAX_SAMPLES_PER_CLASS = None
RETURN_PREDICTIONS = True


In [10]:
def scalar_row(record, variant, corruption, severity):
    row = {"variant": variant, "corruption": corruption, "severity": severity}
    for key, value in record.items():
        if isinstance(value, (int, float, bool, np.floating)):
            row[key] = value
    return row


def condition_loaders():
    for corruption in CORRUPTIONS:
        for severity in SEVERITIES:
            loader, meta = build_imagenet_c_loader(
                DATA_ROOT,
                corruption,
                severity,
                batch_size=BATCH_SIZE,
                num_workers=NUM_WORKERS,
                max_samples_per_class=MAX_SAMPLES_PER_CLASS,
                seed=42,
            )
            yield corruption, severity, loader, meta


In [11]:
resnet50 = make_imagenet_resnet50(device=DEVICE).eval()
dememte_cfg = E6Config(
    dataset="imagenet_c",
    data_dir=str(DATA_ROOT),
    num_classes=1000,
    backbone_name="resnet50",
    backbone_out_channels=2048,
    quantizer_type="ema_vq",
    vq_kmeans_init=False,
    dead_code_restart=False,
)
dememte = make_dememte_variant(dememte_cfg, device=DEVICE).eval()
if DEMEMTE_CHECKPOINT.exists():
    payload = load_checkpoint(dememte, DEMEMTE_CHECKPOINT, device=DEVICE, strict=True)
    print(f"Loaded DeMemte-ImageNet checkpoint: {DEMEMTE_CHECKPOINT}")
    print({k: payload.get(k) for k in ("epoch", "best_val_acc", "codebook_initialized")})
else:
    print(f"WARNING: missing DeMemte-ImageNet checkpoint: {DEMEMTE_CHECKPOINT}")
    print("Train on clean ImageNet first; E10 variants would otherwise use a random head/codebook.")
dememte.eval()


Loaded DeMemte-ImageNet checkpoint: /home/nakato/projects/Dememte/experiments/imagenet_dememte/out/dememte_imagenet_resnet50_vqsa_best.pt
{'epoch': 10, 'best_val_acc': 0.614, 'codebook_initialized': True}


DeMemteVQSA(
  (backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (do

In [12]:
variants = {
    "dememte_imagenet_source": None,
    "e10_assoc_recall_imagenet_c": HippocampalConfig(
        recall_sem=True, recall_epi=False, T=1, beta=1.0, lambda_max=0.1, gate_mode="const"
    ),
    "e10_episodic_imagenet_c": HippocampalConfig(
        recall_sem=False, recall_epi=True, T=1, beta=0.0, lambda_max=0.1, gate_mode="const"
    ),
    "e10_dual_memory_imagenet_c": HippocampalConfig(
        recall_sem=True, recall_epi=True, T=1, beta=0.5, lambda_max=0.1, gate_mode="unfamiliarity"
    ),
}


In [13]:
rows = []
curve_rows = []
prediction_rows = []

for corruption, severity, loader, meta in condition_loaders():
    print(f"Evaluating {corruption}/{severity} ({meta['size']} samples)")
    source = evaluate_baseline(
        resnet50,
        loader,
        device=DEVICE,
        return_predictions=RETURN_PREDICTIONS,
    )
    rows.append(scalar_row(source, "source_resnet50_imagenet", corruption, severity))
    curve_rows.append(scalar_row(source, "source_resnet50_imagenet", corruption, severity))
    if RETURN_PREDICTIONS:
        for item in source.pop("predictions", []):
            prediction_rows.append({"variant": "source_resnet50_imagenet", **item})

    dememte_source = evaluate_dememte(
        dememte,
        loader,
        device=DEVICE,
        return_predictions=RETURN_PREDICTIONS,
    )
    rows.append(scalar_row(dememte_source, "dememte_imagenet_source", corruption, severity))
    curve_rows.append(scalar_row(dememte_source, "dememte_imagenet_source", corruption, severity))
    if RETURN_PREDICTIONS:
        for item in dememte_source.pop("predictions", []):
            prediction_rows.append({"variant": "dememte_imagenet_source", **item})

    for name, cfg in variants.items():
        if cfg is None:
            continue
        adapter = HippocampalMemoryAdapter(dememte, cfg)
        record = evaluate_dememte_tta(
            adapter,
            loader,
            device=DEVICE,
            return_predictions=RETURN_PREDICTIONS,
            tta_method="e10_hippocampal_memory",
            tta_base_variant=name,
        )
        rows.append(scalar_row(record, name, corruption, severity))
        curve_rows.append(scalar_row(record, name, corruption, severity))
        if RETURN_PREDICTIONS:
            for item in record.pop("predictions", []):
                prediction_rows.append({"variant": name, **item})


Evaluating gaussian_noise/3 (50000 samples)
Evaluating gaussian_noise/5 (50000 samples)
Evaluating motion_blur/3 (50000 samples)
Evaluating motion_blur/5 (50000 samples)
Evaluating pixelate/3 (50000 samples)
Evaluating pixelate/5 (50000 samples)
Evaluating jpeg_compression/3 (50000 samples)
Evaluating jpeg_compression/5 (50000 samples)


In [14]:
results = pd.DataFrame(rows)
curves = pd.DataFrame(curve_rows)
preds = pd.DataFrame(prediction_rows)

results.to_csv(OUT_DIR / "e10_imagenet_c_results.csv", index=False)
curves.to_csv(OUT_DIR / "e10_imagenet_c_curves.csv", index=False)
if len(preds):
    preds.to_csv(OUT_DIR / "e10_imagenet_c_predictions.csv", index=False)

summary = []
summary.append("# E10 ImageNet-C Summary\n")
summary.append(f"- conditions: {len(CORRUPTIONS) * len(SEVERITIES)}")
summary.append(f"- device: {DEVICE}")
summary.append(f"- max_samples_per_class: {MAX_SAMPLES_PER_CLASS}")
summary.append("\n## Corrupt Accuracy Avg\n")
if "acc" in results:
    summary.append(results.groupby("variant")["acc"].mean().sort_values(ascending=False).to_markdown())
summary.append("\n## Codebook / Memory Signals\n")
signal_cols = [
    "hard_usage_mean", "dead_code_fraction_mean", "hard_perplexity_mean",
    "dq_mean_mean", "assignment_entropy_mean", "codebook_perplexity_mean",
    "recall_sharpness_mean", "completion_amount_mean", "g_mean_mean",
    "episodic_buffer_churn_mean",
]
available = [c for c in signal_cols if c in results.columns]
if available:
    summary.append(results.groupby("variant")[available].mean().to_markdown())
(OUT_DIR / "e10_imagenet_c_summary.md").write_text("\n".join(summary), encoding="utf-8")
results


,variant,corruption,severity,acc,ece,nll,brier,aurc_confidence,vq_loss_mean,vq_loss_p05,...,g_mean_p50,g_mean_p95,traj_max_step_mean,traj_max_step_p05,traj_max_step_p50,traj_max_step_p95,episodic_buffer_churn_mean,episodic_buffer_churn_p05,episodic_buffer_churn_p50,episodic_buffer_churn_p95
0,source_resnet50_imagenet,gaussian_noise,3,0.44300,0.208621,3.112818,0.758469,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,dememte_imagenet_source,gaussian_noise,3,0.24928,0.083294,4.289258,0.860563,0.496327,0.071621,0.055308,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,e10_assoc_recall_imagenet_c,gaussian_noise,3,0.24956,0.083532,4.290554,0.860383,0.495796,0.071621,0.055308,...,1.000000,1.000000,0.320985,0.291478,0.319401,0.358609,0.000000,0.000000,0.000000,0.000000
3,e10_episodic_imagenet_c,gaussian_noise,3,0.24964,0.081288,4.288830,0.860217,0.496000,0.071621,0.055308,...,1.000000,1.000000,0.292207,0.263688,0.290351,0.325227,0.248175,0.152344,0.261719,0.324219
4,e10_dual_memory_imagenet_c,gaussian_noise,3,0.24954,0.082436,4.289338,0.860259,0.495949,0.071621,0.055308,...,0.999975,1.000000,0.292036,0.266628,0.291102,0.323393,0.211954,0.117188,0.226562,0.281250
5,source_resnet50_imagenet,gaussian_noise,5,0.09644,0.014418,5.640295,0.967433,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,dememte_imagenet_source,gaussian_noise,5,0.04138,0.022249,6.164929,0.987245,0.877780,0.050998,0.037761,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,e10_assoc_recall_imagenet_c,gaussian_noise,5,0.04152,0.022231,6.166119,0.987164,0.877427,0.050998,0.037761,...,1.000000,1.000000,0.271349,0.243575,0.271239,0.298821,0.000000,0.000000,0.000000,0.000000
8,e10_episodic_imagenet_c,gaussian_noise,5,0.04100,0.020668,6.162588,0.987026,0.878453,0.050998,0.037761,...,1.000000,1.000000,0.216616,0.179671,0.217882,0.250543,0.122321,0.058594,0.136719,0.160156
9,e10_dual_memory_imagenet_c,gaussian_noise,5,0.04126,0.021474,6.164286,0.987079,0.877934,0.050998,0.037761,...,0.999388,0.999999,0.227657,0.193674,0.228022,0.260014,0.117657,0.066406,0.125000,0.164062
